# 05 — Реальные shock episodes, категориальная интерпретация и external audit

## Цель

После `04_shock_detector_final.ipynb` имеются **975 месячных сигналов**
в **546 муниципальных рядах**.

В этом notebook мы:

1. объединяем соседние сигналы одного направления в `shock episode`;
2. строим post-hoc интерпретацию по пяти категориям расходов;
3. сравниваем категории строго в `peak_month` общего episode;
4. фиксируем TOP-40 по статистическому `priority_rank`;
5. заранее фиксируем первые TOP-10 для внешней проверки;
6. проводим ручной grouped audit TOP-10 по независимым публичным источникам;
7. сохраняем аналитические и evidence-артефакты.

## Ограничения

Пять компонентных категорий не являются строгим аддитивным разложением
`Все категории`, поэтому используется термин **статистический драйвер**,
а не «денежный вклад».

External evidence не участвует в detector score, выборе порога,
`priority_rank`, TOP-10 или категориальной атрибуции.
Совпадение внешнего события и shock не является доказательством причинности.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

# Portable project root (launch from sb/ or sb/notebooks/).
CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "data" / "interim").is_dir():
    PROJECT_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / "data" / "interim").is_dir():
    PROJECT_DIR = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Запустите notebook из корня sb/ или из sb/notebooks/."
    )
INTERIM_DIR = PROJECT_DIR / "data" / "interim"

# Optional configuration is validated, not used to change frozen results.
CONFIG_FILE = PROJECT_DIR / "configs" / "final_config.yaml"
if CONFIG_FILE.exists():
    try:
        import yaml
    except ImportError:
        print("config.yaml обнаружен; для сверки установите PyYAML")
    else:
        with CONFIG_FILE.open(encoding="utf-8") as fh:
            FINAL_CONFIG = yaml.safe_load(fh) or {}
        configured_n = FINAL_CONFIG.get("external_validation", {}).get("top_n", 10)
        assert int(configured_n) == 10, "Frozen external TOP-10 cannot be changed"
        print("Optional config check: PASSED")

SIGNALS_FILE = INTERIM_DIR / "shock_detector_signals_2024.parquet"
SCORED_PANEL_FILE = INTERIM_DIR / "shock_detector_scored_panel_2024.parquet"
ALL_CATEGORIES_FILE = INTERIM_DIR / "consumption_all_categories_panel_12096.parquet"
REGISTRY_FILE = INTERIM_DIR / "consumption_complete_series_registry_12096.parquet"

required = [
    SIGNALS_FILE,
    SCORED_PANEL_FILE,
    ALL_CATEGORIES_FILE,
    REGISTRY_FILE,
]

missing = [str(p) for p in required if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Не найдены входные файлы:\n"
        + "\n".join(missing)
    )

print("Входные файлы: OK")


In [ ]:
real_signals = pd.read_parquet(SIGNALS_FILE)
real_panel = pd.read_parquet(SCORED_PANEL_FILE)
all_categories = pd.read_parquet(ALL_CATEGORIES_FILE)
complete_registry = pd.read_parquet(REGISTRY_FILE)

for df in [real_signals, real_panel, all_categories]:
    if "period" in df.columns:
        df["period"] = pd.to_datetime(df["period"])

for df in [real_signals, real_panel]:
    if "origin" in df.columns:
        df["origin"] = pd.to_datetime(df["origin"])

print("Месячных сигналов:", len(real_signals))
print("МО с сигналами:", real_signals["series_id"].nunique())
print("Detector panel:", real_panel.shape)
print("6-category panel:", all_categories.shape)
print("Registry:", complete_registry.shape)

assert len(real_signals) == 975
assert real_signals["series_id"].nunique() == 546
assert len(real_panel) == 18_144
assert len(all_categories) == 290_304
assert len(complete_registry) == 12_096

print("Загрузка: PASSED")


# Часть A. Формирование shock episodes

## Правило episode

Сигналы относятся к одному episode, если:

- `series_id` одинаков;
- направление одинаково;
- месяцы идут непосредственно друг за другом.

При смене направления или разрыве хотя бы на один месяц начинается новый episode.
Это строгое правило: пропуски не объединяются вручную.

In [ ]:
signal_df = (
    real_signals
    .sort_values(["series_id", "period"])
    .copy()
)

signal_df["month_index"] = (
    signal_df["period"].dt.year * 12
    + signal_df["period"].dt.month
)

signal_df["prev_month_index"] = (
    signal_df
    .groupby("series_id")["month_index"]
    .shift(1)
)

signal_df["prev_direction"] = (
    signal_df
    .groupby("series_id")["direction"]
    .shift(1)
)

signal_df["month_gap"] = (
    signal_df["month_index"]
    - signal_df["prev_month_index"]
)

signal_df["new_episode"] = (
    signal_df["prev_month_index"].isna()
    | (signal_df["month_gap"] != 1)
    | (signal_df["direction"] != signal_df["prev_direction"])
)

signal_df["episode_number"] = (
    signal_df
    .groupby("series_id")["new_episode"]
    .cumsum()
    .astype(int)
)

signal_df["episode_id"] = (
    signal_df["series_id"].astype(str)
    + "|episode_"
    + signal_df["episode_number"].astype(str)
)

assert not signal_df.duplicated(["series_id", "period"]).any()

print("Сигналов:", len(signal_df))
print("Episodes:", signal_df["episode_id"].nunique())


In [ ]:
episode_core = (
    signal_df
    .groupby(
        ["episode_id", "series_id", "mo"],
        as_index=False,
    )
    .agg(
        direction=("direction", "first"),
        size_group=("size_group", "first"),
        start_month=("period", "min"),
        end_month=("period", "max"),
        n_signal_months=("period", "size"),
        max_score_ratio=("score_ratio", "max"),
        mean_score_ratio=("score_ratio", "mean"),
        max_abs_robust_score=(
            "robust_score",
            lambda x: x.abs().max(),
        ),
        max_abs_surprise_pct=(
            "surprise_pct",
            lambda x: x.abs().max(),
        ),
    )
)

peak_idx = (
    signal_df
    .groupby("episode_id")["score_ratio"]
    .idxmax()
)

peak_rows = (
    signal_df.loc[
        peak_idx,
        [
            "episode_id",
            "period",
            "surprise_pct",
            "robust_score",
            "score_ratio",
            "actual_value",
            "prediction",
        ],
    ]
    .rename(
        columns={
            "period": "peak_month",
            "surprise_pct": "peak_surprise_pct",
            "robust_score": "peak_robust_score",
            "score_ratio": "peak_score_ratio",
            "actual_value": "peak_actual_value",
            "prediction": "peak_prediction",
        }
    )
)

shock_episodes = (
    episode_core
    .merge(
        peak_rows,
        on="episode_id",
        how="left",
        validate="one_to_one",
    )
)

# Triage score: не вероятность реального события.
shock_episodes["score_pct_rank"] = (
    shock_episodes["max_score_ratio"]
    .rank(pct=True)
)

shock_episodes["magnitude_pct_rank"] = (
    shock_episodes["max_abs_surprise_pct"]
    .rank(pct=True)
)

shock_episodes["duration_pct_rank"] = (
    shock_episodes["n_signal_months"]
    .rank(pct=True)
)

shock_episodes["priority_score"] = (
    shock_episodes[
        [
            "score_pct_rank",
            "magnitude_pct_rank",
            "duration_pct_rank",
        ]
    ]
    .mean(axis=1)
)

shock_episodes["priority_rank"] = (
    shock_episodes["priority_score"]
    .rank(
        ascending=False,
        method="first",
    )
    .astype(int)
)

shock_episodes = (
    shock_episodes
    .sort_values("priority_rank")
    .reset_index(drop=True)
)

print("Shock episodes:", len(shock_episodes))
print("МО с episodes:", shock_episodes["series_id"].nunique())
print(
    "Медианная длительность:",
    shock_episodes["n_signal_months"].median(),
)
print(
    "Медианный peak score ratio:",
    round(
        shock_episodes["max_score_ratio"].median(),
        3,
    ),
)

display(
    shock_episodes[
        [
            "priority_rank",
            "mo",
            "start_month",
            "end_month",
            "direction",
            "n_signal_months",
            "max_score_ratio",
            "max_abs_surprise_pct",
            "priority_score",
        ]
    ]
    .head(20)
    .round(3)
)


In [ ]:
import matplotlib.pyplot as plt
episode_summary = (
    shock_episodes
    .groupby("direction", as_index=False)
    .agg(
        n_episodes=("episode_id", "size"),
        n_municipalities=("series_id", "nunique"),
        median_duration=("n_signal_months", "median"),
        median_peak_score=("max_score_ratio", "median"),
        median_abs_surprise_pct=("max_abs_surprise_pct", "median"),
    )
)

display(episode_summary.round(3))

duration_summary = (
    shock_episodes["n_signal_months"]
    .value_counts()
    .sort_index()
    .rename_axis("n_signal_months")
    .reset_index(name="n_episodes")
)

display(duration_summary)

ax = duration_summary.plot(
    x="n_signal_months",
    y="n_episodes",
    kind="bar",
    legend=False,
    figsize=(9, 4.5),
)

ax.set_title("Длительность shock episodes")
ax.set_xlabel("Число последовательных сигнальных месяцев")
ax.set_ylabel("Число episodes")

plt.tight_layout()
plt.show()


### Вывод

После агрегации единицей анализа становится `shock episode`, а не месячная тревога.

Это уменьшает риск интерпретировать один продолжающийся режим
как несколько независимых экономических событий.
Причина episode на этом этапе всё ещё неизвестна.

# Часть B. Безопасное сопоставление пяти категорий

Название МО допускается к атрибуции только тогда, когда во всех шести категориях
этому названию соответствует ровно один полный временной ряд.

In [ ]:
CATEGORY_SLUGS = {
    "Здоровье": "health",
    "Общественное питание": "foodservice",
    "Продовольствие": "grocery",
    "Маркетплейсы": "marketplace",
    "Транспорт": "transport",
}

COMPONENT_CATEGORIES = list(CATEGORY_SLUGS.keys())

ALL_CATEGORY_NAMES = [
    "Все категории",
    *COMPONENT_CATEGORIES,
]

registry_counts = (
    complete_registry
    .groupby(["mo", "category_15"])
    .size()
    .unstack(fill_value=0)
    .reindex(
        columns=ALL_CATEGORY_NAMES,
        fill_value=0,
    )
)

safe_names = set(
    registry_counts.index[
        registry_counts.eq(1).all(axis=1)
    ]
)

total_registry = (
    complete_registry[
        complete_registry["category_15"]
        == "Все категории"
    ][["series_id", "mo"]]
    .rename(
        columns={
            "series_id": "total_series_id"
        }
    )
    .copy()
)

category_map = total_registry.copy()

category_map["category_alignment_ok"] = (
    category_map["mo"].isin(safe_names)
)

for category, slug in CATEGORY_SLUGS.items():

    safe_cat = (
        complete_registry[
            (complete_registry["category_15"] == category)
            & complete_registry["mo"].isin(safe_names)
        ][["mo", "series_id"]]
        .rename(
            columns={
                "series_id": f"{slug}_series_id"
            }
        )
    )

    assert not safe_cat["mo"].duplicated().any()

    # Слева остаются неоднозначные названия, поэтому many_to_one.
    category_map = category_map.merge(
        safe_cat,
        on="mo",
        how="left",
        validate="many_to_one",
    )

coverage_cols = [
    f"{slug}_series_id"
    for slug in CATEGORY_SLUGS.values()
]

category_map["n_category_matches"] = (
    category_map[coverage_cols]
    .notna()
    .sum(axis=1)
)

n_safe = int(
    category_map["category_alignment_ok"].sum()
)

print("Всего total series:", len(category_map))
print("Безопасно сопоставлено:", n_safe)
print(
    "Покрытие:",
    f"{100 * n_safe / len(category_map):.2f}%",
)

display(
    category_map["n_category_matches"]
    .value_counts()
    .sort_index()
    .rename_axis("n_categories")
    .to_frame("n_series")
)

assert len(category_map) == 2016
assert category_map["total_series_id"].nunique() == 2016

assert (
    category_map.loc[
        category_map["category_alignment_ok"],
        "n_category_matches",
    ]
    == 5
).all()

print("Category mapping: PASSED")


# Часть C. Causal category surprise

Для каждой компонентной категории:

1. one-step прогноз строится только по истории до origin;
2. `surprise = log(actual / forecast)`;
3. размерный квинтиль определяется отдельно внутри категории по 2023 году;
4. surprise переводится в robust z-score внутри `category x size_group x month`.

Категории не используются для изменения detector threshold.

In [ ]:
category_wide = {}

for category in COMPONENT_CATEGORIES:

    wide = (
        all_categories[
            all_categories["category_15"] == category
        ]
        .pivot(
            index="series_id",
            columns="period",
            values="value",
        )
        .sort_index(axis=1)
    )

    wide.columns = pd.to_datetime(wide.columns)

    category_wide[category] = wide

    print(category, wide.shape)

    assert wide.shape == (2016, 24)

print("Category-wide panels: PASSED")


In [ ]:
SIZE_LABELS = ["Q1", "Q2", "Q3", "Q4", "Q5"]

cat_meta_parts = []

for category in COMPONENT_CATEGORIES:

    wide = category_wide[category]

    cols_2023 = [
        c
        for c in wide.columns
        if c.year == 2023
    ]

    mean_2023 = wide[cols_2023].mean(axis=1)

    groups = pd.qcut(
        mean_2023.rank(method="first"),
        q=5,
        labels=SIZE_LABELS,
    )

    cat_meta_parts.append(
        pd.DataFrame({
            "series_id": wide.index.to_numpy(),
            "category_15": category,
            "category_mean_2023": mean_2023.to_numpy(),
            "category_size_group": groups.astype(str).to_numpy(),
        })
    )

cat_meta = pd.concat(
    cat_meta_parts,
    ignore_index=True,
)

display(
    cat_meta
    .groupby(
        ["category_15", "category_size_group"]
    )
    .size()
    .unstack(fill_value=0)
)

assert len(cat_meta) == 2016 * 5


In [ ]:
def seasonal_growth_h1(wide, origin, window=3):
    # Каузальный one-step сезонный прогноз.
    origin = pd.Timestamp(origin)
    target = origin + pd.DateOffset(months=1)

    hist = wide.loc[:, wide.columns <= origin].copy()

    log_hist = np.log(
        hist.clip(lower=1e-6)
    )

    log_yoy = (
        log_hist
        - log_hist.shift(12, axis=1)
    )

    recent_growth = (
        log_yoy.iloc[:, -window:]
        .mean(axis=1)
    )

    seasonal_month = (
        target
        - pd.DateOffset(months=12)
    )

    seasonal_anchor = hist[seasonal_month]

    return (
        seasonal_anchor
        * np.exp(recent_growth)
    )


def robust_z(values):
    values = pd.Series(values)

    med = values.median()

    mad = float(
        np.median(
            np.abs(values - med)
        )
    )

    scale = 1.4826 * mad

    if (
        not np.isfinite(scale)
        or scale <= 1e-12
    ):
        return pd.Series(
            np.zeros(len(values)),
            index=values.index,
        )

    return (values - med) / scale


In [ ]:
EVAL_PERIODS = pd.date_range(
    "2024-04-01",
    "2024-12-01",
    freq="MS",
)

category_score_parts = []

for category in COMPONENT_CATEGORIES:

    print("Категория:", category)

    wide = category_wide[category]

    meta = (
        cat_meta[
            cat_meta["category_15"] == category
        ][
            [
                "series_id",
                "category_size_group",
            ]
        ]
        .copy()
    )

    for period in EVAL_PERIODS:

        origin = (
            period
            - pd.DateOffset(months=1)
        )

        prediction = seasonal_growth_h1(
            wide,
            origin,
            window=3,
        )

        actual = wide[period]

        surprise = np.log(
            actual.clip(lower=1e-6)
            / prediction.clip(lower=1e-6)
        )

        part = pd.DataFrame({
            "series_id": wide.index.to_numpy(),
            "category_15": category,
            "period": period,
            "origin": origin,
            "actual_value": actual.to_numpy(),
            "prediction": prediction.to_numpy(),
            "category_surprise": surprise.to_numpy(),
        })

        part = part.merge(
            meta,
            on="series_id",
            how="left",
            validate="one_to_one",
        )

        part["category_z"] = (
            part
            .groupby(
                "category_size_group",
                group_keys=False,
            )["category_surprise"]
            .transform(robust_z)
        )

        part["category_surprise_pct"] = (
            100.0
            * (
                np.exp(part["category_surprise"])
                - 1.0
            )
        )

        category_score_parts.append(part)

category_scores = pd.concat(
    category_score_parts,
    ignore_index=True,
)

print("Category score rows:", len(category_scores))

assert len(category_scores) == 5 * 2016 * 9
assert category_scores["category_z"].notna().all()

print("Category score: PASSED")


# Часть D. Категориальная атрибуция episode

## 6. Атрибуция строго в peak month общего episode

Для каждого `shock episode` уже определён `peak_month` —
месяц, в котором агрегированный detector имеет максимальный `score_ratio`.

Категориальная интерпретация выполняется **только в этот же месяц**.

Это важно по двум причинам:

1. все пять категорий сравниваются в одной временной точке;
2. длительный episode не получает искусственно больше шансов стать `broad_local`
   только потому, что каждая категория могла быть аномальной в разные месяцы.

Категория считается направленно согласованной, если одновременно:

- знак raw `category_surprise` совпадает с направлением общего episode;
- знак robust `category_z` также совпадает с направлением общего episode.

Категория считается сильной согласованной, если дополнительно:

`abs(category_z) >= 2`

Порог `2` используется только для post-hoc интерпретации.
Он не влияет на detector score, порог детектора или synthetic TEST.

Типы профиля:

- `broad_local` — минимум 3 сильных согласованных категории в peak month;
- `sectoral` — 1–2 сильных согласованных категории;
- `weak/mixed` — безопасное сопоставление есть, но сильной категории нет;
- `unavailable` — межкатегориальное сопоставление неоднозначно.

`main_driver` — сильная согласованная категория с максимальным `abs(category_z)`
в peak month общего episode.

Это статистическая атрибуция, а не доказательство причинности
и не денежный вклад категории.

In [ ]:
# ---------------------------------------------------------
# 1. Для каждого episode берём только peak month
# ---------------------------------------------------------

episode_peaks = (
    shock_episodes[
        [
            "episode_id",
            "series_id",
            "mo",
            "direction",
            "peak_month",
        ]
    ]
    .rename(
        columns={
            "series_id":
                "total_series_id",
            "direction":
                "episode_direction",
        }
    )
    .copy()
)

episode_peaks = (
    episode_peaks
    .merge(
        category_map,
        on=[
            "total_series_id",
            "mo",
        ],
        how="left",
        validate="many_to_one",
    )
)

assert len(
    episode_peaks
) == len(
    shock_episodes
)

assert not episode_peaks[
    "episode_id"
].duplicated().any()

print(
    "Episodes для peak-month attribution:",
    len(
        episode_peaks
    ),
)

print(
    "Episodes с безопасным category alignment:",
    int(
        episode_peaks[
            "category_alignment_ok"
        ]
        .fillna(False)
        .sum()
    ),
)


# ---------------------------------------------------------
# 2. Подключаем пять категорий строго в peak_month
# ---------------------------------------------------------

peak_category_parts = []

for (
    category,
    slug
) in CATEGORY_SLUGS.items():

    series_col = (
        f"{slug}_series_id"
    )

    temp = (
        episode_peaks[
            [
                "episode_id",
                "total_series_id",
                "mo",
                "episode_direction",
                "peak_month",
                "category_alignment_ok",
                series_col,
            ]
        ]
        .rename(
            columns={
                series_col:
                    "category_series_id",
            }
        )
        .copy()
    )

    scores = (
        category_scores[
            category_scores[
                "category_15"
            ]
            == category
        ][
            [
                "series_id",
                "period",
                "category_surprise",
                "category_surprise_pct",
                "category_z",
                "category_size_group",
            ]
        ]
        .rename(
            columns={
                "series_id":
                    "category_series_id",
                "period":
                    "peak_month",
            }
        )
        .copy()
    )

    temp = (
        temp
        .merge(
            scores,
            on=[
                "category_series_id",
                "peak_month",
            ],
            how="left",
            validate="many_to_one",
        )
    )

    temp[
        "category_15"
    ] = category

    peak_category_parts.append(
        temp
    )

episode_category_peak = (
    pd.concat(
        peak_category_parts,
        ignore_index=True,
    )
)

episode_category_peak[
    "episode_sign"
] = np.where(
    episode_category_peak[
        "episode_direction"
    ]
    == "positive",
    1,
    -1,
)

episode_category_peak[
    "direction_match"
] = (
    episode_category_peak[
        "category_z"
    ]
    .notna()
    &
    (
        np.sign(
            episode_category_peak[
                "category_z"
            ]
        )
        == episode_category_peak[
            "episode_sign"
        ]
    )
    &
    (
        np.sign(
            episode_category_peak[
                "category_surprise"
            ]
        )
        == episode_category_peak[
            "episode_sign"
        ]
    )
)

episode_category_peak[
    "strong_aligned"
] = (
    episode_category_peak[
        "direction_match"
    ]
    &
    (
        episode_category_peak[
            "category_z"
        ]
        .abs()
        >= 2.0
    )
)

assert (
    episode_category_peak
    .groupby(
        "episode_id"
    )
    .size()
    .eq(5)
    .all()
)

print(
    "Peak-month category rows:",
    len(
        episode_category_peak
    ),
)

print(
    "Peak-month attribution table: PASSED"
)


In [ ]:
# ---------------------------------------------------------
# 3. Одна строка на episode x category
# ---------------------------------------------------------

category_episode_summary = (
    episode_category_peak[
        [
            "episode_id",
            "category_15",
            "category_alignment_ok",
            "peak_month",
            "category_surprise",
            "category_surprise_pct",
            "category_z",
            "direction_match",
            "strong_aligned",
            "category_size_group",
        ]
    ]
    .copy()
)

category_episode_summary[
    "abs_category_z"
] = (
    category_episode_summary[
        "category_z"
    ]
    .abs()
)

assert (
    category_episode_summary
    .groupby(
        "episode_id"
    )
    .size()
    .eq(5)
    .all()
)

assert not category_episode_summary.duplicated(
    [
        "episode_id",
        "category_15",
    ]
).any()

print(
    "Episode-category summary: PASSED"
)

display(
    category_episode_summary
    .head(10)
    .round(3)
)


In [ ]:
# ---------------------------------------------------------
# 4. Тип профиля и main driver в peak month
# ---------------------------------------------------------

attribution_rows = []

for (
    episode_id,
    part
) in (
    category_episode_summary
    .groupby(
        "episode_id",
        sort=False,
    )
):

    alignment_ok = bool(
        part[
            "category_alignment_ok"
        ]
        .fillna(False)
        .all()
    )

    strong = (
        part[
            part[
                "strong_aligned"
            ]
        ]
        .sort_values(
            "abs_category_z",
            ascending=False,
        )
        .copy()
    )

    n_strong = int(
        len(
            strong
        )
    )

    if not alignment_ok:

        profile_type = (
            "unavailable"
        )

        main_driver = None
        main_driver_z = np.nan
        main_driver_surprise_pct = np.nan
        supporting_categories = None

    elif n_strong >= 3:

        profile_type = (
            "broad_local"
        )

        main = (
            strong.iloc[0]
        )

        main_driver = (
            main[
                "category_15"
            ]
        )

        main_driver_z = float(
            main[
                "category_z"
            ]
        )

        main_driver_surprise_pct = float(
            main[
                "category_surprise_pct"
            ]
        )

        supporting_categories = (
            "; ".join(
                strong[
                    "category_15"
                ].tolist()
            )
        )

    elif n_strong >= 1:

        profile_type = (
            "sectoral"
        )

        main = (
            strong.iloc[0]
        )

        main_driver = (
            main[
                "category_15"
            ]
        )

        main_driver_z = float(
            main[
                "category_z"
            ]
        )

        main_driver_surprise_pct = float(
            main[
                "category_surprise_pct"
            ]
        )

        supporting_categories = (
            "; ".join(
                strong[
                    "category_15"
                ].tolist()
            )
        )

    else:

        profile_type = (
            "weak/mixed"
        )

        main_driver = None
        main_driver_z = np.nan
        main_driver_surprise_pct = np.nan
        supporting_categories = None

    attribution_rows.append({
        "episode_id":
            episode_id,

        "category_alignment_ok":
            alignment_ok,

        "profile_type":
            profile_type,

        "n_interpretable_categories":
            n_strong,

        "main_driver":
            main_driver,

        "main_driver_z":
            main_driver_z,

        "main_driver_surprise_pct":
            main_driver_surprise_pct,

        "supporting_categories":
            supporting_categories,
    })

episode_attribution = pd.DataFrame(
    attribution_rows
)

episodes_enriched = (
    shock_episodes
    .merge(
        episode_attribution,
        on="episode_id",
        how="left",
        validate="one_to_one",
    )
)

assert len(
    episodes_enriched
) == len(
    shock_episodes
)

assert episodes_enriched[
    "profile_type"
].notna().all()

print(
    "Peak-month категориальная атрибуция: PASSED"
)


In [ ]:
profile_summary = (
    episodes_enriched[
        "profile_type"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "profile_type"
    )
    .reset_index(
        name="n_episodes"
    )
)

profile_summary[
    "share_pct"
] = (
    100.0
    * profile_summary[
        "n_episodes"
    ]
    / len(
        episodes_enriched
    )
)

print(
    "Типы профиля — только по peak month:"
)

display(
    profile_summary
    .round(2)
)

driver_summary = (
    episodes_enriched[
        episodes_enriched[
            "main_driver"
        ]
        .notna()
    ][
        "main_driver"
    ]
    .value_counts()
    .rename_axis(
        "main_driver"
    )
    .reset_index(
        name="n_episodes"
    )
)

print(
    "Главные статистические драйверы — только по peak month:"
)

display(
    driver_summary
)

n_interpretable = int(
    episodes_enriched[
        "main_driver"
    ]
    .notna()
    .sum()
)

print(
    "Episodes с main driver:",
    n_interpretable,
    "из",
    len(
        episodes_enriched
    ),
)


In [ ]:
# TOP-40 выбирается по заранее рассчитанному priority_rank.
# Категориальная атрибуция не влияет на этот ranking.

top40 = (
    episodes_enriched
    .sort_values(
        "priority_rank"
    )
    .head(40)
    .copy()
)

print(
    "TOP-40 episodes с peak-month attribution:"
)

display(
    top40[
        [
            "priority_rank",
            "mo",
            "start_month",
            "end_month",
            "peak_month",
            "direction",
            "n_signal_months",
            "max_score_ratio",
            "max_abs_surprise_pct",
            "profile_type",
            "n_interpretable_categories",
            "main_driver",
            "main_driver_z",
        ]
    ]
    .round(3)
)

top40_profile = (
    top40[
        "profile_type"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "profile_type"
    )
    .reset_index(
        name="n"
    )
)

top40_driver = (
    top40[
        top40[
            "main_driver"
        ]
        .notna()
    ][
        "main_driver"
    ]
    .value_counts()
    .rename_axis(
        "main_driver"
    )
    .reset_index(
        name="n"
    )
)

print(
    "TOP-40: profile type"
)

display(
    top40_profile
)

print(
    "TOP-40: main driver"
)

display(
    top40_driver
)


# ---------------------------------------------------------
# Диагностика: сколько сильных категорий одновременно
# присутствует именно в peak month.
# ---------------------------------------------------------

top40_strength = (
    top40[
        "n_interpretable_categories"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "n_strong_categories_at_peak"
    )
    .reset_index(
        name="n_episodes"
    )
)

print(
    "TOP-40: число сильных категорий в peak month"
)

display(
    top40_strength
)


### Вывод по атрибуции

Категориальный слой является только **post-hoc interpretation layer**.

Он не участвует в:

- detector score;
- выборе порога;
- synthetic TEST;
- расчёте Event Recall;
- формировании `priority_rank`.

Все пять категорий теперь сравниваются **строго в `peak_month` общего episode**.

Это устраняет важный источник смещения:
длительный episode больше не получает дополнительные шансы стать `broad_local`
только потому, что разные категории могли быть сильными в разные месяцы.

`main_driver` означает наиболее сильное статистически согласованное
категориальное отклонение **в момент максимального общего shock**.

Большое значение robust z означает сильное положение относительно
`category x size_group x month`, но не должно интерпретироваться как
«столько стандартных отклонений» в параметрической нормальной модели.

Категориальная атрибуция остаётся описательной.
Она не является доказательством причины episode.

# Часть E. Предварительно зафиксированная external validation

Внешняя проверка начинается только после фиксации статистических результатов.

Для audit заранее берутся первые **10 episodes по `priority_rank`**.
Список case study не меняется после просмотра внешних источников.

Поисковое окно:

`один месяц до начала episode -> один месяц после окончания episode`

Допустимые статусы:

- `strong_match`;
- `possible_match`;
- `external_event_without_clear_link`;
- `no_clear_explanation`.

In [ ]:
EXTERNAL_EVIDENCE_N = 10

external_validation_cases = (
    top40
    .sort_values("priority_rank")
    .head(EXTERNAL_EVIDENCE_N)
    .copy()
)

evidence_template = (
    external_validation_cases[
        [
            "episode_id",
            "priority_rank",
            "series_id",
            "mo",
            "start_month",
            "end_month",
            "peak_month",
            "direction",
            "n_signal_months",
            "max_score_ratio",
            "max_abs_surprise_pct",
            "profile_type",
            "n_interpretable_categories",
            "main_driver",
            "main_driver_z",
            "supporting_categories",
        ]
    ]
    .copy()
)

evidence_template["search_start"] = (
    evidence_template["start_month"]
    - pd.DateOffset(months=1)
)

evidence_template["search_end"] = (
    evidence_template["end_month"]
    + pd.DateOffset(months=1)
)

for col in [
    "evidence_status",
    "external_event_date",
    "event_title",
    "source_name",
    "source_url",
    "notes",
]:
    evidence_template[col] = pd.Series(
        "",
        index=evidence_template.index,
        dtype="object",
    )

assert len(evidence_template) == 10
assert (
    evidence_template["priority_rank"].astype(int).tolist()
    == list(range(1, 11))
)

print(
    "External-validation cases зафиксированы:",
    len(evidence_template),
)

with pd.option_context(
    "display.max_colwidth", None,
    "display.width", 400,
    "display.max_columns", None,
):
    display(
        evidence_template[
            [
                "priority_rank",
                "mo",
                "start_month",
                "end_month",
                "peak_month",
                "direction",
                "profile_type",
                "main_driver",
                "search_start",
                "search_end",
            ]
        ]
    )


## E.1 Протокол ручного audit

TOP-10 разбивается на три группы заранее:

- **Group A — structural / administrative:** ranks 1, 4, 6;
- **Group B — local economic / infrastructure:** ranks 2, 5, 7, 9;
- **Group C — unresolved / difficult:** ranks 3, 8, 10.

Для каждого case проверяются:

1. точность географии;
2. попадание во временное окно;
3. качество источника;
4. согласованность с направлением shock;
5. согласованность с категориальным профилем;
6. итоговый evidence-status.

Если точного объяснения нет, сохраняется `no_clear_explanation`.
Live web retrieval и Media Cloud в финальном notebook не используются.

In [ ]:
AUDIT_GROUPS = {
    "A_structural": [1, 4, 6],
    "B_local_economic": [2, 5, 7, 9],
    "C_unresolved": [3, 8, 10],
}

all_group_ranks = [
    rank
    for ranks in AUDIT_GROUPS.values()
    for rank in ranks
]

assert sorted(all_group_ranks) == list(range(1, 11))
assert len(all_group_ranks) == len(set(all_group_ranks))

rank_to_group = {
    rank: group
    for group, ranks in AUDIT_GROUPS.items()
    for rank in ranks
}

manual_audit = (
    evidence_template[
        [
            "episode_id",
            "priority_rank",
            "mo",
            "start_month",
            "end_month",
            "peak_month",
            "direction",
            "n_signal_months",
            "max_score_ratio",
            "max_abs_surprise_pct",
            "profile_type",
            "n_interpretable_categories",
            "main_driver",
            "main_driver_z",
            "supporting_categories",
            "search_start",
            "search_end",
        ]
    ]
    .copy()
)

manual_audit["audit_group"] = (
    manual_audit["priority_rank"]
    .map(rank_to_group)
)

assert manual_audit["audit_group"].notna().all()

display(
    manual_audit[
        [
            "priority_rank",
            "mo",
            "audit_group",
            "direction",
            "profile_type",
            "main_driver",
            "peak_month",
        ]
    ]
)


## E.2 Зафиксированный registry проверенных внешних фактов

Ниже хранится результат ручной проверки TOP-10.

Для ретроспективных источников точный месяц события не придумывается:
если источник подтверждает проблему за 2024 год, но не позволяет определить
её начало внутри конкретного месяца, `external_event_date` остаётся пустой,
а статус не повышается выше `possible_match`.

In [ ]:
evidence_records = [
    {
        "priority_rank": 1,
        "external_event_date": "2024-05-08",
        "event_type": "administrative_reform",
        "source_class": "official_municipal",
        "source_name": "Муниципальный округ Бекасово — историческая справка",
        "source_url": "https://mobekasovo.ru/city/history.php",
        "event_summary": (
            "Законом Москвы от 8 мая 2024 года муниципальный округ Бекасово "
            "образован в ходе административно-территориальных преобразований "
            "с включением поселения Киевский."
        ),
        "geo_match": "exact",
        "time_match": "inside_episode",
        "direction_consistency": "not_testable",
        "category_consistency": "broad_structural_pattern",
        "evidence_status": "strong_match",
        "review_notes": (
            "Episode апрель–август, peak — июнь. "
            "Реформа произошла внутри episode. "
            "Все 5 категорий имеют сильный согласованный сигнал. "
            "Это strong structural match, а не доказанный causal spending event."
        ),
    },
    {
        "priority_rank": 2,
        "external_event_date": "2024-07-20",
        "event_type": "city_event",
        "source_class": "local_press",
        "source_name": "Большая районка — День города Мамоново",
        "source_url": (
            "https://kaliningrad-vremya.ru/wp-content/uploads/2026/02/"
            "bolshaya_rayonka_no12_55_25_iyulya_2024.pdf"
        ),
        "event_summary": (
            "20 июля 2024 года в Мамоново прошёл День города "
            "с праздничными, культурными и спортивными мероприятиями."
        ),
        "geo_match": "exact",
        "time_match": "inside_episode_far_from_peak",
        "direction_consistency": "weak_positive_plausibility",
        "category_consistency": "weak_for_broad_persistent_shock",
        "evidence_status": "external_event_without_clear_link",
        "review_notes": (
            "Episode июль–сентябрь, peak — сентябрь. "
            "Краткое июльское событие плохо объясняет устойчивый "
            "broad_local shock по пяти категориям."
        ),
    },
    {
        "priority_rank": 3,
        "external_event_date": "",
        "event_type": "unresolved",
        "source_class": "",
        "source_name": "",
        "source_url": "",
        "event_summary": (
            "В фиксированном окне март–июнь 2024 не установлено локального "
            "события достаточной силы и качества источника, которое убедительно "
            "объясняло бы положительный episode Старошайговского района."
        ),
        "geo_match": "not_established",
        "time_match": "not_established",
        "direction_consistency": "not_testable",
        "category_consistency": "unresolved",
        "evidence_status": "no_clear_explanation",
        "review_notes": (
            "Shock очень большой по magnitude, но профиль sectoral: "
            "Продовольствие и Маркетплейсы. Общерегиональные события "
            "не используются как доказательство для конкретного МО."
        ),
    },
    {
        "priority_rank": 4,
        "external_event_date": "2024-05-08",
        "event_type": "administrative_reform",
        "source_class": "official_municipal",
        "source_name": "Муниципальный округ Внуково — история района",
        "source_url": "https://www.munvnukovo.ru/mun-1/munokr/istoriya-rayona/",
        "event_summary": (
            "Закон Москвы №13 от 8 мая 2024 года изменил границы "
            "муниципального округа Внуково с включением территории "
            "поселения Кокошкино."
        ),
        "geo_match": "exact",
        "time_match": "inside_episode",
        "direction_consistency": "not_testable",
        "category_consistency": "broad_structural_pattern",
        "evidence_status": "strong_match",
        "review_notes": (
            "Episode май–сентябрь, peak — июль. Реформа произошла "
            "в начале episode. Все 5 категорий имеют сильный сигнал. "
            "Это структурное совпадение, но не доказательство причинности."
        ),
    },
    {
        "priority_rank": 5,
        "external_event_date": "2024-10-23",
        "event_type": "infrastructure_completion",
        "source_class": "regional_media",
        "source_name": "АСТВ — Новости Сахалинской области",
        "source_url": (
            "https://www.astv.ru/news/society/"
            "2024-10-23-v-makarove-gotovyat-k-otkrytiyu-"
            "sportkompleks-s-ledovoj-arenoj"
        ),
        "event_summary": (
            "В октябре 2024 года в Макарове готовился к открытию "
            "универсальный спорткомплекс с ледовой ареной; "
            "строительные работы находились на финальной стадии."
        ),
        "geo_match": "exact",
        "time_match": "near_peak",
        "direction_consistency": "plausible_positive",
        "category_consistency": "partial",
        "evidence_status": "possible_match",
        "review_notes": (
            "Episode октябрь–ноябрь, peak — октябрь. "
            "Объект находится в точной географии и близко к peak month. "
            "Связь правдоподобна, но причинность не доказана."
        ),
    },
    {
        "priority_rank": 6,
        "external_event_date": "2024-05-08",
        "event_type": "administrative_reform",
        "source_class": "official_law",
        "source_name": "Закон города Москвы от 08.05.2024 №13",
        "source_url": (
            "https://www.mos.ru/upload/documents/docs/343/"
            "Zakon13-ajbn2.pdf"
        ),
        "event_summary": (
            "Закон №13 изменил муниципальное устройство Новой Москвы; "
            "территория бывшего поселения Щаповское была распределена "
            "между новыми муниципальными образованиями."
        ),
        "geo_match": "exact",
        "time_match": "inside_episode",
        "direction_consistency": "not_testable",
        "category_consistency": "broad_structural_pattern",
        "evidence_status": "strong_match",
        "review_notes": (
            "Episode апрель–июнь, peak — июнь. "
            "Все 5 категорий сильные. Case важен как возможный "
            "structural break географически агрегированного ряда."
        ),
    },
    {
        "priority_rank": 7,
        "external_event_date": "2024-12-15",
        "event_type": "infrastructure_construction",
        "source_class": "project_operator",
        "source_name": "Архангельский Экологический Оператор",
        "source_url": (
            "https://aeo29.ru/%D0%BC%D1%83%D1%81%D0%BE%D1%80%D0%BE"
            "%D1%81%D0%BE%D1%80%D1%82%D0%B8%D1%80%D0%BE%D0%B2%D0%BE"
            "%D1%87%D0%BD%D1%8B%D0%B9-%D0%BA%D0%BE%D0%BC%D0%BF%D0%BB"
            "%D0%B5%D0%BA%D1%81-%D0%B2-%D0%BA%D0%BE%D1%80%D1%8F/"
        ),
        "event_summary": (
            "К середине декабря 2024 года мусоросортировочный комплекс "
            "в Коряжме достиг примерно 70% готовности; продолжались "
            "строительство, лицензирование и подбор более 100 сотрудников."
        ),
        "geo_match": "exact",
        "time_match": "near_peak",
        "direction_consistency": "plausible_positive",
        "category_consistency": "broad_but_indirect",
        "evidence_status": "possible_match",
        "review_notes": (
            "Episode ноябрь–декабрь, peak — декабрь. "
            "Проект совпадает по времени с shock, но начался раньше. "
            "15 декабря — техническая дата для формулировки источника "
            "'на середину декабря'."
        ),
    },
    {
        "priority_rank": 8,
        "external_event_date": "",
        "event_type": "public_sector_wage_arrears_retrospective",
        "source_class": "official_court_retrospective",
        "source_name": "Бай-Тайгинский районный суд Республики Тыва",
        "source_url": (
            "https://bai-taiginskiy--tva.sudrf.ru/modules.php"
            "?case_id=99450706"
            "&case_uid=88745386-30a6-4dc8-b996-b8367d95cae2"
            "&delo_id=1540005"
            "&name=sud_delo"
            "&name_op=case"
            "&srv_num=1"
        ),
        "event_summary": (
            "Позднейшие судебные материалы подтверждают задолженности "
            "по заработной плате работников школы Бай-Тайгинского района "
            "за периоды, включающие 2024 год."
        ),
        "geo_match": "exact",
        "time_match": "2024_overlap_not_month_specific",
        "direction_consistency": "plausible_negative",
        "category_consistency": "partial_indirect",
        "evidence_status": "possible_match",
        "review_notes": (
            "Episode октябрь–ноябрь отрицательный. Задолженность по зарплате "
            "экономически согласуется с возможным снижением активности, "
            "но ретроспективный источник не позволяет установить начало "
            "проблемы именно в октябре–ноябре 2024. Поэтому event date пустая."
        ),
    },
    {
        "priority_rank": 9,
        "external_event_date": "2024-06-29",
        "event_type": "city_anniversary",
        "source_class": "official_regional",
        "source_name": "Костромская областная Дума",
        "source_url": (
            "https://www.kosoblduma.ru/press/article/"
            "Dvoiinoii_iubileii_01_07_2024.html"
        ),
        "event_summary": (
            "В конце июня 2024 года Волгореченск отметил 60-летие города "
            "и 55-летие пуска первого энергоблока Костромской ГРЭС."
        ),
        "geo_match": "exact",
        "time_match": "inside_episode_near_peak",
        "direction_consistency": "inconsistent",
        "category_consistency": "inconsistent_with_negative_shock",
        "evidence_status": "external_event_without_clear_link",
        "review_notes": (
            "Episode май–июль отрицательный, peak — июль. "
            "Массовое мероприятие находится рядом с peak, "
            "но плохо согласуется с широким снижением расходов."
        ),
    },
    {
        "priority_rank": 10,
        "external_event_date": "",
        "event_type": "unresolved",
        "source_class": "",
        "source_name": "",
        "source_url": "",
        "event_summary": (
            "В фиксированном окне июнь–сентябрь 2024 не установлено "
            "локального подтверждённого события, которое убедительно "
            "объясняло бы отрицательный broad_local episode "
            "Котласского муниципального округа."
        ),
        "geo_match": "not_established",
        "time_match": "not_established",
        "direction_consistency": "not_testable",
        "category_consistency": "unresolved",
        "evidence_status": "no_clear_explanation",
        "review_notes": (
            "Отдельные природные риски и инфраструктурные новости "
            "не дают достаточного основания для причинной интерпретации."
        ),
    },
]

verified_evidence = pd.DataFrame(evidence_records)

assert len(verified_evidence) == 10
assert verified_evidence["priority_rank"].is_unique
assert sorted(
    verified_evidence["priority_rank"].astype(int).tolist()
) == list(range(1, 11))

manual_audit = (
    manual_audit
    .merge(
        verified_evidence,
        on="priority_rank",
        how="left",
        validate="one_to_one",
    )
)

assert manual_audit["evidence_status"].notna().all()

print(
    "External evidence заполнен:",
    len(verified_evidence),
    "cases",
)


## E.3 Group A — structural / administrative

Ranks 1, 4 и 6 образуют отдельный structural-кластер:

- Киевский;
- Кокошкино;
- Щаповское.

Все три cases имеют `broad_local` профиль с пятью сильными категориями
и совпадают по времени с административно-территориальной реформой Москвы
от 8 мая 2024 года.

Они классифицируются как `strong_match`, но не как доказанный
причинный эффект реформы на расходы.

In [ ]:
group_a_review = (
    manual_audit[
        manual_audit["audit_group"]
        == "A_structural"
    ]
    .sort_values("priority_rank")
    .copy()
)

assert len(group_a_review) == 3
assert group_a_review["priority_rank"].tolist() == [1, 4, 6]
assert group_a_review["evidence_status"].eq("strong_match").all()
assert group_a_review["geo_match"].eq("exact").all()
assert group_a_review["time_match"].eq("inside_episode").all()

print("GROUP A review: PASSED")

with pd.option_context(
    "display.max_colwidth", None,
    "display.width", 500,
    "display.max_columns", None,
):
    display(
        group_a_review[
            [
                "priority_rank",
                "mo",
                "peak_month",
                "direction",
                "profile_type",
                "n_interpretable_categories",
                "main_driver",
                "event_type",
                "external_event_date",
                "evidence_status",
                "source_class",
                "source_name",
            ]
        ]
    )

    print("\nGROUP A — источники и reasoning:")

    display(
        group_a_review[
            [
                "priority_rank",
                "mo",
                "event_summary",
                "source_url",
                "review_notes",
            ]
        ]
    )


## E.4 Group B — local economic / infrastructure

Ranks 2, 5, 7 и 9 проверяют более сложные локальные события.

Итог:

- Мамоновский — `external_event_without_clear_link`;
- Макаровский — `possible_match`;
- Коряжма — `possible_match`;
- Волгореченск — `external_event_without_clear_link`.

Ни один case этой группы не повышается до `strong_match`.

In [ ]:
group_b_review = (
    manual_audit[
        manual_audit["audit_group"]
        == "B_local_economic"
    ]
    .sort_values("priority_rank")
    .copy()
)

assert len(group_b_review) == 4
assert group_b_review["priority_rank"].tolist() == [2, 5, 7, 9]
assert group_b_review["evidence_status"].notna().all()
assert group_b_review["source_url"].ne("").all()
assert group_b_review["geo_match"].eq("exact").all()

print("GROUP B review: PASSED")

display(
    group_b_review["evidence_status"]
    .value_counts()
    .rename_axis("evidence_status")
    .reset_index(name="n_cases")
)

with pd.option_context(
    "display.max_colwidth", None,
    "display.width", 500,
    "display.max_columns", None,
):
    display(
        group_b_review[
            [
                "priority_rank",
                "mo",
                "peak_month",
                "direction",
                "profile_type",
                "main_driver",
                "event_type",
                "external_event_date",
                "time_match",
                "direction_consistency",
                "evidence_status",
                "source_name",
            ]
        ]
    )

    print("\nGROUP B — источники и reasoning:")

    display(
        group_b_review[
            [
                "priority_rank",
                "mo",
                "event_summary",
                "source_url",
                "review_notes",
            ]
        ]
    )


## E.5 Group C — unresolved / difficult

Ranks 3, 8 и 10 используются как проверка устойчивости интерпретации.

- Старошайговский — `no_clear_explanation`;
- Бай-Тайгинский — осторожный `possible_match` на основании
  ретроспективного официального судебного источника;
- Котласский — `no_clear_explanation`.

Для Бай-Тайгинского точный месяц начала проблемы не устанавливается
из доступного судебного материала, поэтому точная event date не задаётся.

In [ ]:
group_c_review = (
    manual_audit[
        manual_audit["audit_group"]
        == "C_unresolved"
    ]
    .sort_values("priority_rank")
    .copy()
)

assert len(group_c_review) == 3
assert group_c_review["priority_rank"].tolist() == [3, 8, 10]
assert group_c_review["evidence_status"].notna().all()

matched_c = (
    group_c_review[
        group_c_review["evidence_status"]
        != "no_clear_explanation"
    ]
)

assert matched_c["source_url"].ne("").all()

print("GROUP C review: PASSED")

display(
    group_c_review["evidence_status"]
    .value_counts()
    .rename_axis("evidence_status")
    .reset_index(name="n_cases")
)

with pd.option_context(
    "display.max_colwidth", None,
    "display.width", 500,
    "display.max_columns", None,
):
    display(
        group_c_review[
            [
                "priority_rank",
                "mo",
                "peak_month",
                "direction",
                "profile_type",
                "main_driver",
                "event_type",
                "external_event_date",
                "time_match",
                "direction_consistency",
                "evidence_status",
                "source_name",
            ]
        ]
    )

    print("\nGROUP C — reasoning:")

    display(
        group_c_review[
            [
                "priority_rank",
                "mo",
                "event_summary",
                "source_url",
                "review_notes",
            ]
        ]
    )


## E.6 Финальный audit TOP-10

Все десять заранее выбранных cases остаются в анализе,
включая случаи без ясного внешнего объяснения.

Это позволяет отделить:

`statistical detection -> category interpretation -> external evidence`

In [ ]:
final_manual_audit = (
    manual_audit
    .sort_values("priority_rank")
    .reset_index(drop=True)
    .copy()
)

ALLOWED_FINAL_STATUSES = {
    "strong_match",
    "possible_match",
    "external_event_without_clear_link",
    "no_clear_explanation",
}

EXPECTED_STATUS_COUNTS = {
    "strong_match": 3,
    "possible_match": 3,
    "external_event_without_clear_link": 2,
    "no_clear_explanation": 2,
}

assert len(final_manual_audit) == 10
assert final_manual_audit["priority_rank"].tolist() == list(range(1, 11))
assert final_manual_audit["evidence_status"].notna().all()

assert set(
    final_manual_audit["evidence_status"]
).issubset(ALLOWED_FINAL_STATUSES)

matched_mask = (
    final_manual_audit["evidence_status"]
    != "no_clear_explanation"
)

assert final_manual_audit.loc[
    matched_mask,
    "source_name",
].ne("").all()

assert final_manual_audit.loc[
    matched_mask,
    "source_url",
].ne("").all()

observed_status_counts = (
    final_manual_audit["evidence_status"]
    .value_counts()
    .to_dict()
)

assert observed_status_counts == EXPECTED_STATUS_COUNTS

evidence_status_summary = (
    final_manual_audit["evidence_status"]
    .value_counts()
    .rename_axis("evidence_status")
    .reset_index(name="n_cases")
)

evidence_status_summary["share_pct"] = (
    100.0
    * evidence_status_summary["n_cases"]
    / len(final_manual_audit)
)

print("FINAL TOP-10 AUDIT: PASSED")

display(
    evidence_status_summary.round(1)
)

with pd.option_context(
    "display.max_colwidth", None,
    "display.width", 500,
    "display.max_columns", None,
):
    display(
        final_manual_audit[
            [
                "priority_rank",
                "mo",
                "audit_group",
                "peak_month",
                "direction",
                "profile_type",
                "main_driver",
                "event_type",
                "external_event_date",
                "geo_match",
                "time_match",
                "direction_consistency",
                "category_consistency",
                "evidence_status",
                "source_class",
                "source_name",
            ]
        ]
    )


## E.7 News / external-event alignment protocol

Внешние факты приведены к единой схеме `event_id × municipality × event_month`.
Это **не дополнительная обучающая выборка**: десять событий отобраны для
ретроспективной проверки, **после** фиксации `priority_rank`. Их нельзя использовать
для обучения/оценки forecasting на тех же муниципалитетах без независимого
проспективного корпуса новостей и отдельного walk-forward теста.

Храним раздельно дату события (`event_date`), дату публикации (`publication_date`)
и месяц события (`event_month`). В исходном ручном реестре дата публикации не
документирована; поэтому она остаётся `NaT`, и такие записи **запрещены** как
признаки на исторических forecast origins. URL (в том числе с датами в пути)
не считается достаточным подтверждением даты публикации. Нет события или
подтверждённой даты — нет временного признака.

`mo` используется здесь только для интерпретационного соединения с фиксированным
TOP-10. Для будущего масштабирования требуется уникальный `series_id` / проверенный
geo crosswalk, проверка географических омонимов и отделение региона от МО.
Статусы `strong_match` и `possible_match` **не являются причинными метками**,
а процент внешних соответствий **не является precision детектора**.


In [ ]:
# Structured, retrospective TOP-10 event registry. No new factual claims.
# The manual TOP-10 audit omits series_id. Restore it from the frozen
# episode registry by the unique episode_id, never by municipality name.
assert shock_episodes["episode_id"].is_unique
assert final_manual_audit["episode_id"].is_unique
assert "series_id" not in final_manual_audit.columns

episode_identity = shock_episodes[["episode_id", "series_id", "mo"]].copy()
news_events = final_manual_audit.merge(
    episode_identity[["episode_id", "series_id", "mo"]].rename(
        columns={"mo": "episode_registry_mo"}
    ),
    on="episode_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)
assert len(news_events) == len(final_manual_audit) == 10
assert news_events["_merge"].eq("both").all()
assert news_events["series_id"].notna().all()
assert news_events["mo"].eq(news_events["episode_registry_mo"]).all()
news_events = news_events.drop(columns=["episode_registry_mo", "_merge"])
print("TOP-10 episode identity mapping: PASSED")
news_events["event_id"] = news_events["priority_rank"].map(
    lambda k: f"audit2024_top10_{int(k):02d}"
)
news_events["event_date"] = pd.to_datetime(
    news_events["external_event_date"].replace("", pd.NA), errors="coerce"
)
# Publication dates were not recorded in the frozen manual audit.
news_events["publication_date"] = pd.Series(pd.NaT, index=news_events.index, dtype="datetime64[ns]")
news_events["event_month"] = news_events["event_date"].dt.to_period("M").dt.to_timestamp()
news_events["publication_date_verified"] = False
news_events["is_retrospective_audit"] = True
news_events["source_confidence"] = pd.NA  # not rated; do not fabricate a score
news_events = news_events.rename(columns={"direction":"shock_direction"})
EVENT_FIELDS = [
    "event_id", "episode_id", "priority_rank", "series_id", "mo",
    "event_date", "event_month", "publication_date", "publication_date_verified",
    "event_type", "source_class", "source_name", "source_url",
    "source_confidence", "shock_direction", "category_consistency",
    "geo_match", "time_match", "direction_consistency", "evidence_status",
    "is_retrospective_audit", "event_summary",
]
news_events = news_events[EVENT_FIELDS].copy()
assert len(news_events) == 10 and news_events["event_id"].is_unique
assert news_events["publication_date"].isna().all()
assert news_events["episode_id"].is_unique

# Month-level evidence: count only documented external events with a dated event.
# This is retrospective description, NOT a forecast-ready news feature.
known_events = news_events.loc[news_events["event_month"].notna()].copy()
news_monthly_retrospective = (
    known_events.groupby(["series_id", "mo", "event_month"], as_index=False)
    .agg(
        news_count=("event_id", "nunique"),
        official_news_count=("source_class", lambda s: int(s.fillna("").str.startswith("official").sum())),
        strong_evidence_count=("evidence_status", lambda s: int(s.eq("strong_match").sum())),
        possible_evidence_count=("evidence_status", lambda s: int(s.eq("possible_match").sum())),
    )
)
assert not news_monthly_retrospective.duplicated(["series_id", "event_month"]).any()

# A point-in-time-safe *interface*: rows without verified publication dates
# are excluded; the retrospective TOP-10 is ineligible regardless of date.
def news_available_at_origin(events: pd.DataFrame, origin) -> pd.DataFrame:
    origin = pd.Timestamp(origin)
    published = pd.to_datetime(events["publication_date"], errors="coerce")
    valid = (
        events["publication_date_verified"].eq(True)
        & ~events["is_retrospective_audit"].eq(True)
        & published.notna()
        & published.le(origin)
    )
    return events.loc[valid].copy()

# Contract tests: future/unknown publication excluded; verified historical
# and prospective record accepted. The synthetic sample is never used in model fitting.
test_news = pd.DataFrame({
    "publication_date": [pd.Timestamp("2024-05-01"), pd.Timestamp("2024-08-01"), pd.NaT],
    "publication_date_verified": [True, True, False],
    "is_retrospective_audit": [False, False, False],
})
assert len(news_available_at_origin(test_news, "2024-06-01")) == 1
assert news_available_at_origin(news_events, "2024-06-01").empty
print("Time-safe publication cutoff contract: PASSED")
print("Retrospective audit events:", len(news_events))
print("Dated retrospective event-months:", len(news_monthly_retrospective))
print("Forecast-eligible audit events: 0 (publication dates not verified)")
display(news_monthly_retrospective)


## E.8 Количественный News Alignment Audit (ретроспективный)

Дополнительная диагностическая оценка выполняется **только** по заранее зафиксированным 10 случаям. Она не добавляет новых новостей, не изменяет TOP-10 и не использует информацию из внешних источников для прогноза или детектора.

Метрики различают **наличие подтверждённого источника**, **проверку географического, временного и направленного соответствия**, **доступность даты события** и **доступность даты публикации**. Пропуск или неподтверждённая проверка не считаются успешным совпадением. Статусы внешнего соответствия описательны; они не являются оценкой precision детектора.


In [ ]:
# Quantitative retrospective news alignment audit — frozen TOP-10 only.
# No fitting, news enrichment, forecast adjustment or detector updates.
assert len(news_events) == 10
assert news_events["event_id"].is_unique
assert news_events["series_id"].notna().all()

_news = news_events.copy()
_news["has_source"] = (
    _news["source_url"].fillna("").astype(str).str.strip().ne("")
    & _news["source_name"].fillna("").astype(str).str.strip().ne("")
)
_news["dated_event"] = _news["event_date"].notna()
_news["verified_publication"] = (
    _news["publication_date_verified"].eq(True)
    & _news["publication_date"].notna()
)
# Existing audit fields can contain booleans, flags or descriptive labels.
# Report their raw distributions instead of guessing an ordinal score.
CHECK_COLS = ["geo_match", "time_match", "direction_consistency", "category_consistency"]
news_alignment_check_distributions = pd.concat(
    [
        _news[c].fillna("<missing>").astype(str)
        .value_counts(dropna=False)
        .rename_axis("recorded_value")
        .reset_index(name="n_records")
        .assign(check=c)
        for c in CHECK_COLS
    ], ignore_index=True,
)[["check", "recorded_value", "n_records"]]

news_alignment_quantitative = pd.DataFrame([
    {"metric":"frozen_top10_episodes", "value":len(_news)},
    {"metric":"records_with_source", "value":int(_news["has_source"].sum())},
    {"metric":"records_with_event_date", "value":int(_news["dated_event"].sum())},
    {"metric":"dated_series_months", "value":len(news_monthly_retrospective)},
    {"metric":"verified_publication_dates", "value":int(_news["verified_publication"].sum())},
    {"metric":"forecast_eligible_audit_records", "value":len(news_available_at_origin(_news, "2024-12-01"))},
    *[{"metric":f"status_{status}","value":int(_news["evidence_status"].eq(status).sum())}
      for status in sorted(ALLOWED_FINAL_STATUSES)],
])
assert not news_alignment_quantitative["metric"].duplicated().any()
assert int(news_alignment_quantitative.loc[
    news_alignment_quantitative["metric"].eq("forecast_eligible_audit_records"),"value"
].iloc[0]) == 0
assert news_alignment_quantitative.loc[
    news_alignment_quantitative["metric"].str.startswith("status_"),"value"
].sum() == 10
print("Quantitative retrospective News Alignment Audit: PASSED")
display(news_alignment_quantitative)
display(news_alignment_check_distributions)


# Часть F. Финальная сводка

Компактный evidence layer формируется из полного ручного audit,
после чего собирается единая таблица результатов notebook 05.

In [ ]:
completed_evidence = (
    final_manual_audit[
        [
            "episode_id",
            "evidence_status",
            "external_event_date",
            "event_summary",
            "source_name",
            "source_url",
            "review_notes",
        ]
    ]
    .rename(
        columns={
            "event_summary": "event_title",
            "review_notes": "notes",
        }
    )
    .copy()
)

assert len(completed_evidence) == 10
assert completed_evidence["episode_id"].is_unique

n_safe_episodes = int(
    episodes_enriched["category_alignment_ok"]
    .fillna(False)
    .sum()
)

n_with_driver = int(
    episodes_enriched["main_driver"]
    .notna()
    .sum()
)

status_counts = (
    completed_evidence["evidence_status"]
    .value_counts()
    .to_dict()
)

n_external_cases = len(completed_evidence)
n_strong = int(status_counts.get("strong_match", 0))
n_possible = int(status_counts.get("possible_match", 0))
n_external_no_link = int(
    status_counts.get(
        "external_event_without_clear_link",
        0,
    )
)
n_no_explanation = int(
    status_counts.get(
        "no_clear_explanation",
        0,
    )
)

summary_05 = pd.DataFrame([
    {"metric": "monthly_signals", "value": len(real_signals)},
    {
        "metric": "series_with_signals",
        "value": real_signals["series_id"].nunique(),
    },
    {"metric": "shock_episodes", "value": len(shock_episodes)},
    {
        "metric": "series_with_episodes",
        "value": shock_episodes["series_id"].nunique(),
    },
    {
        "metric": "safe_category_alignment_episodes",
        "value": n_safe_episodes,
    },
    {
        "metric": "episodes_with_main_driver",
        "value": n_with_driver,
    },
    {
        "metric": "top40_broad_local",
        "value": int(
            (top40["profile_type"] == "broad_local").sum()
        ),
    },
    {
        "metric": "top40_sectoral",
        "value": int(
            (top40["profile_type"] == "sectoral").sum()
        ),
    },
    {
        "metric": "top40_weak_mixed",
        "value": int(
            (top40["profile_type"] == "weak/mixed").sum()
        ),
    },
    {
        "metric": "external_validation_cases",
        "value": n_external_cases,
    },
    {
        "metric": "external_strong_match",
        "value": n_strong,
    },
    {
        "metric": "external_possible_match",
        "value": n_possible,
    },
    {
        "metric": "external_event_without_clear_link",
        "value": n_external_no_link,
    },
    {
        "metric": "external_no_clear_explanation",
        "value": n_no_explanation,
    },
    {
        "metric": "external_strong_match_pct",
        "value": 100.0 * n_strong / n_external_cases,
    },
    {
        "metric": "external_strong_or_possible_pct",
        "value": (
            100.0
            * (n_strong + n_possible)
            / n_external_cases
        ),
    },
    {
        "metric": "external_event_identified_pct",
        "value": (
            100.0
            * (
                n_strong
                + n_possible
                + n_external_no_link
            )
            / n_external_cases
        ),
    },
])

print("FINAL NOTEBOOK 05 SUMMARY")

display(summary_05.round(2))

print("\nПрофили всех episodes:")
display(profile_summary.round(2))

print("\nГлавные статистические драйверы:")
display(driver_summary)

print("\nПрофили TOP-40:")
display(top40_profile)

print("\nДрайверы TOP-40:")
display(top40_driver)

print("\nExternal evidence TOP-10:")
display(evidence_status_summary.round(1))


# Часть G. Сохранение финальных артефактов

Сохраняются:

- месячные сигналы с episode-id;
- таблица episodes;
- категориальные scores;
- итоговая категориальная атрибуция;
- TOP-40;
- template TOP-10;
- полный manual audit;
- компактный completed evidence;
- итоговая summary;
- manifest.

Финальный pipeline не требует внешнего API.

In [ ]:
OUTPUT_FILES = {
    "episode_signals":
        INTERIM_DIR / "shock_episode_monthly_signals_2024.parquet",
    "episodes":
        INTERIM_DIR / "shock_episodes_2024.parquet",
    "category_scores":
        INTERIM_DIR / "shock_category_scores_2024.parquet",
    "category_episode_summary":
        INTERIM_DIR / "shock_episode_category_scores.parquet",
    "episodes_enriched":
        INTERIM_DIR / "shock_episodes_attributed_2024.parquet",
    "top40":
        INTERIM_DIR / "shock_top40_episodes.csv",
    "evidence_template":
        INTERIM_DIR / "shock_external_evidence_template.csv",
    "manual_audit":
        INTERIM_DIR / "shock_external_evidence_manual_audit.csv",
    "completed_evidence":
        INTERIM_DIR / "shock_external_evidence_completed.csv",
    "summary":
        INTERIM_DIR / "shock_real_cases_summary.csv",
    "news_events": INTERIM_DIR / "shock_news_events_aligned_top10.csv",
    "news_monthly": INTERIM_DIR / "shock_news_monthly_retrospective_top10.csv",
    "news_audit_quantitative": INTERIM_DIR / "shock_news_alignment_quantitative_top10.csv",
    "news_audit_checks": INTERIM_DIR / "shock_news_alignment_checks_top10.csv",
    "manifest":
        INTERIM_DIR / "shock_real_cases_manifest.json",
}

signal_df.to_parquet(
    OUTPUT_FILES["episode_signals"],
    index=False,
)

shock_episodes.to_parquet(
    OUTPUT_FILES["episodes"],
    index=False,
)

category_scores.to_parquet(
    OUTPUT_FILES["category_scores"],
    index=False,
)

category_episode_summary.to_parquet(
    OUTPUT_FILES["category_episode_summary"],
    index=False,
)

episodes_enriched.to_parquet(
    OUTPUT_FILES["episodes_enriched"],
    index=False,
)

top40.to_csv(
    OUTPUT_FILES["top40"],
    index=False,
)

evidence_template.to_csv(
    OUTPUT_FILES["evidence_template"],
    index=False,
)

final_manual_audit.to_csv(
    OUTPUT_FILES["manual_audit"],
    index=False,
)

completed_evidence.to_csv(
    OUTPUT_FILES["completed_evidence"],
    index=False,
)

summary_05.to_csv(
    OUTPUT_FILES["summary"],
    index=False,
)

news_events.to_csv(OUTPUT_FILES["news_events"], index=False)
news_monthly_retrospective.to_csv(OUTPUT_FILES["news_monthly"], index=False)
news_alignment_quantitative.to_csv(OUTPUT_FILES["news_audit_quantitative"], index=False)
news_alignment_check_distributions.to_csv(OUTPUT_FILES["news_audit_checks"], index=False)

manifest = {
    "name":
        "Real shock episodes, category attribution and external audit",

    "input_monthly_signals":
        int(len(real_signals)),

    "input_series_with_signals":
        int(real_signals["series_id"].nunique()),

    "n_episodes":
        int(len(shock_episodes)),

    "n_series_with_episodes":
        int(shock_episodes["series_id"].nunique()),

    "episode_rule":
        (
            "consecutive detected months within the same total series "
            "and the same direction"
        ),

    "category_alignment":
        (
            "municipality name is used only if it maps to exactly one "
            "complete series in each of all six categories"
        ),

    "n_safe_total_series":
        int(n_safe),

    "category_forecast":
        "causal seasonal-growth h1, YoY window=3",

    "category_normalization":
        (
            "robust z inside category x "
            "2023-size-quintile x month"
        ),

    "category_attribution_time":
        (
            "all five component categories are evaluated only "
            "at aggregate episode peak_month"
        ),

    "strong_category_rule":
        (
            "at peak_month, raw surprise and robust z "
            "have episode direction and abs(z) >= 2"
        ),

    "priority_score":
        (
            "mean percentile rank of peak score ratio, "
            "absolute surprise magnitude and episode duration; "
            "triage only"
        ),

    "top_n_saved": 40,

    "external_validation": {
        "n_cases": 10,
        "selection":
            (
                "first 10 episodes by frozen priority_rank; "
                "selected before external source review"
            ),
        "audit_groups": AUDIT_GROUPS,
        "strong_match": n_strong,
        "possible_match": n_possible,
        "external_event_without_clear_link":
            n_external_no_link,
        "no_clear_explanation":
            n_no_explanation,
        "strong_match_pct":
            round(
                100.0 * n_strong / n_external_cases,
                2,
            ),
        "strong_or_possible_pct":
            round(
                100.0
                * (n_strong + n_possible)
                / n_external_cases,
                2,
            ),
        "protocol":
            (
                "manual grouped audit of frozen TOP-10 "
                "using independent public sources; "
                "no live retrieval inside notebook"
            ),
        "review_date": "2026-10-07",
    },

    "news_alignment": {
        "registry": "TOP-10 retrospective external-event audit",
        "event_grain": "event_id, series_id, event_month",
        "forecast_origin_rule": "verified publication_date <= origin; exclude retrospective audit",
        "publication_dates_verified": 0,
        "forecast_model_uses_news": False,
        "detector_uses_news": False,
        "retrospective_monthly_events": int(len(known_events)),
        "quantitative_audit_file": OUTPUT_FILES["news_audit_quantitative"].name,
        "check_distribution_file": OUTPUT_FILES["news_audit_checks"].name,
    },

    "causality_note":
        (
            "category attribution and external evidence are "
            "descriptive associations and do not establish causality"
        ),
}

with open(
    OUTPUT_FILES["manifest"],
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2,
    )

# Round-trip проверка сохранённого evidence.
completed_saved = pd.read_csv(
    OUTPUT_FILES["completed_evidence"]
)

expected_saved = (
    completed_evidence
    .fillna("")
    .sort_values("episode_id")
    .reset_index(drop=True)
)

observed_saved = (
    completed_saved
    .fillna("")
    .sort_values("episode_id")
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    expected_saved,
    observed_saved,
    check_dtype=False,
)

artifact_status = pd.DataFrame([
    {
        "artifact": path.name,
        "exists": path.exists(),
    }
    for path in OUTPUT_FILES.values()
])

display(artifact_status)

assert artifact_status["exists"].all()

print("FINAL EXTERNAL EVIDENCE: PASSED")
print("FINAL NOTEBOOK 05 ARTIFACTS: PASSED")
print("Артефакты сохранены в:", INTERIM_DIR)


# Итог notebook 05

Из **975 месячных сигналов** сформировано **826 shock episodes**
в 546 муниципальных рядах.

Для **776 episodes** доступно безопасное межкатегориальное сопоставление,
а в **639 cases** обнаружена хотя бы одна сильная согласованная категория.

Структура всех episodes:

- `sectoral` — **474 (57.38%)**;
- `broad_local` — **165 (19.98%)**;
- `weak/mixed` — **137 (16.59%)**;
- `unavailable` — **50 (6.05%)**.

Среди TOP-40:

- 28 — `broad_local`;
- 10 — `sectoral`;
- 2 — `weak/mixed`.

## External validation

Для внешней проверки заранее зафиксированы первые 10 episodes
по `priority_rank`.

Ручной grouped audit дал:

- **3 / 10 — `strong_match`**;
- **3 / 10 — `possible_match`**;
- **2 / 10 — `external_event_without_clear_link`**;
- **2 / 10 — `no_clear_explanation`**.

Таким образом, **60% TOP-10** имеют сильное или правдоподобное
внешнее соответствие, но только **30%** классифицированы как `strong_match`.

Отдельно выявлен structural-класс:
Киевский, Кокошкино и Щаповское совпадают по времени
с административно-территориальной реформой Москвы
и имеют `broad_local` профиль с пятью сильными категориями.

Для Бай-Тайгинского сохранён только осторожный `possible_match`:
ретроспективный официальный судебный источник подтверждает задолженности
по зарплате за периоды, включающие 2024 год, но не позволяет надёжно
привязать начало проблемы к конкретному месяцу shock episode.

**Shock Radar следует трактовать как систему обнаружения и приоритизации
статистических аномалий, а не как автоматическую систему определения причин.**

External evidence используется только для независимой post-hoc проверки
и интерпретации и не участвует в настройке detector.

## News Alignment Layer — результат и ограничения

Сформированы воспроизводимые таблицы `shock_news_events_aligned_top10.csv`
и `shock_news_monthly_retrospective_top10.csv`. Они содержат структуру
проверенных внешних фактов и их ретроспективную привязку к МО/месяцу.
Фактические `publication_date` в ручном реестре не подтверждены, поэтому
эти 10 записей **не используются как исторические forecasting-признаки**.
Добавлена проверенная на контрактных примерах функция point-in-time фильтрации
для будущего независимого корпуса новостей с подтверждёнными датами публикации.
Детектор, веса, TEST и исходный TOP-10 остались неизменными.
